In [1]:
import os
import duckdb
os.chdir('..')
con = duckdb.connect('data/transit_kpi.duckdb')

In [2]:
#from google.transit import gtfs_realtime_pb2'
#from datetime import datetime, timezone
# Load the protobuf library's GTFS-RT message definition
#feed = gtfs_realtime_pb2.FeedMessage()


#with open('data/raw/gtfs_rt/20260408T204522Z.pb', 'rb') as f:
    #feed.ParseFromString(f.read())


#print(f"Feed timestamp: {datetime.fromtimestamp(feed.header.timestamp, tz=timezone.utc)}")
#print(f"GTFS-RT version: {feed.header.gtfs_realtime_version}")
#print(f"Number of entities: {len(feed.entity)}")

In [3]:
# Grab the first entity and print its raw structure
#entity = feed.entity[0]
#print(entity)

In [4]:
# Get the first stop update from this entity
#stop_update = entity.trip_update.stop_time_update[0]

# Convert Unix timestamp to readable datetime
#predicted_arrival = datetime.fromtimestamp(
#    stop_update.arrival.time, 
#    tz=timezone.utc
#)
#
#print(f"Trip ID: {entity.trip_update.trip.trip_id}")
#print(f"Route ID: {entity.trip_update.trip.route_id}")
#print(f"Stop sequence: {stop_update.stop_sequence}")
#print(f"Stop ID: {stop_update.stop_id}")
#print(f"Predicted arrival (UTC): {predicted_arrival}")
#print(f"Predicted arrival (Eastern): {predicted_arrival.astimezone()}")

In [5]:
import os
from pathlib import Path

# Point to local folder of .pb files
archive_dir = Path('data/raw/gtfs_rt')

# Get all .pb files and extract their timestamps from the filename
pb_files = sorted(archive_dir.glob('*.pb'))

print(f"Total files: {len(pb_files)}")
print(f"First file: {pb_files[0].name}")
print(f"Last file:  {pb_files[-1].name}")

Total files: 288
First file: 20260407T000452Z.pb
Last file:  20260408T204522Z.pb


In [6]:
import duckdb
from pathlib import Path
from google.transit import gtfs_realtime_pb2
from datetime import datetime, timezone


archive_dir = Path('data/raw/gtfs_rt')

# Get all .pb files sorted chronologically
pb_files = sorted(archive_dir.glob('*.pb'))

print(f"Files to process: {len(pb_files)}")

Files to process: 288


#Parser

#Parser

In [7]:
records = []
con.execute("DROP TABLE IF EXISTS gtfsrt")

for pb_file in pb_files:
    
    # Pull timestamp from filename e.g. 20260407T143022Z.pb
    snapshot_ts = datetime.strptime(
        pb_file.stem,
        "%Y%m%dT%H%M%SZ"
    ).replace(tzinfo=timezone.utc)
    service_date = snapshot_ts.date().isoformat()
    
    feed = gtfs_realtime_pb2.FeedMessage()
    with open(pb_file, 'rb') as f:
        feed.ParseFromString(f.read())
    
    for entity in feed.entity:
        
        if not entity.HasField('trip_update'):
            continue
        
        tu = entity.trip_update
        trip_id = tu.trip.trip_id
        route_id = tu.trip.route_id
        schedule_relationship = tu.trip.schedule_relationship
        
        for stu in tu.stop_time_update:
            
            # Prefer arrival time, fall back to departure
            if stu.HasField('arrival'):
                predicted_time = stu.arrival.time
            elif stu.HasField('departure'):
                predicted_time = stu.departure.time
            else:
                continue
            
            records.append({
                'snapshot_ts':           snapshot_ts.isoformat(),
                'service_date':          service_date,
                'trip_id':               trip_id,
                'route_id':              route_id,
                'schedule_relationship': schedule_relationship,
                'stop_sequence':         stu.stop_sequence,
                'stop_id':               stu.stop_id,
                'predicted_unix':        predicted_time,
                'predicted_ts':          datetime.fromtimestamp(
                                             predicted_time,
                                             tz=timezone.utc
                                         ).isoformat()
            })

print(f"Total records: {len(records)}")

Total records: 4390409


In [8]:
import pandas as pd

# Convert list of dicts to a pandas DataFrame
df = pd.DataFrame(records)

# Load into DuckDB as a table called 'gtfsrt'
con.execute("DROP TABLE IF EXISTS gtfsrt")
con.execute("CREATE TABLE gtfsrt AS SELECT * FROM df")

# Sanity check
con.sql("SELECT COUNT(*) as total_rows FROM gtfsrt").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows
0,4390409


In [9]:
con.sql("SELECT snapshot_ts, service_date, trip_id, route_id FROM gtfsrt LIMIT 5").df()

,snapshot_ts,service_date,trip_id,route_id
0,2026-04-07T00:04:52+00:00,2026-04-07,701387,103
1,2026-04-07T00:04:52+00:00,2026-04-07,701387,103
2,2026-04-07T00:04:52+00:00,2026-04-07,701387,103
3,2026-04-07T00:04:52+00:00,2026-04-07,701387,103
4,2026-04-07T00:04:52+00:00,2026-04-07,701387,103


In [10]:
con.sql("""
    SELECT 
        route_id,
        route_short_name,
        route_long_name
    FROM read_csv_auto('data/google_bus/routes.txt')
    WHERE route_short_name IN ('23', '47')
""").df()

,route_id,route_short_name,route_long_name
0,23,23,11th Market to Chestnut Hill
1,47,47,Whitman Plaza to 5th-Godfrey


In [11]:
con.sql("""
    SELECT
        route_id,
        COUNT(DISTINCT trip_id) as unique_trips,
        COUNT(DISTINCT snapshot_ts) as snapshots,
        COUNT(*) as total_records
    FROM gtfsrt
    WHERE route_id IN ('23', '47')
    GROUP BY route_id
    ORDER BY route_id
""").df()

,route_id,unique_trips,snapshots,total_records
0,23,221,288,138366
1,47,210,288,136526


In [12]:
con.sql("""
    -- Load stop_times into DuckDB as a permanent table
      DROP TABLE IF EXISTS stop_times;
    
    CREATE TABLE stop_times AS
    SELECT
        trip_id,
        stop_sequence,
        stop_id,
        arrival_time,
        departure_time,
        -- Convert GTFS time string to seconds since midnight
        -- Same conversion we did in notebook 01
        CAST(SPLIT_PART(arrival_time, ':', 1) AS INTEGER) * 3600 +
        CAST(SPLIT_PART(arrival_time, ':', 2) AS INTEGER) * 60 +
        CAST(SPLIT_PART(arrival_time, ':', 3) AS INTEGER) AS arrival_seconds
    FROM read_csv_auto('data/google_bus/stop_times.txt')
""")

con.sql("SELECT COUNT(*) as rows FROM stop_times").df()

,rows
0,2045404
